In [ ]:
# ── CELL 1: INSTALL & IMPORTS ──────────────────────────────────────────────
!pip install -q python-telegram-bot transformers accelerate torch

import asyncio
import re
import logging
from datetime import datetime
from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, filters, ContextTypes
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from kaggle_secrets import UserSecretsClient

# disable telegram logging spam
logging.basicConfig(level=logging.CRITICAL)
for logger_name in ["httpx", "httpcore", "telegram", "apscheduler"]:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)

print("✓ imports loaded")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2: LOAD QWEN (SAVVY)
# ══════════════════════════════════════════════════════════════════════════════

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

QWEN_MODEL = "qwen/qwen3.5-4B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[device] {DEVICE}")
print(f"[loading savvy] {QWEN_MODEL}")

savvy_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL,
    trust_remote_code=True
)

savvy_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

savvy_model.eval()
print(f"✓ Savvy (Qwen) loaded on {DEVICE}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3: LOAD TINY AYA (WVY)
# ══════════════════════════════════════════════════════════════════════════════

AYA_MODEL = "/kaggle/input/models/cohere-labs/tiny-aya/transformers/global/1"

print(f"[loading wvy] {AYA_MODEL}")

wvy_tokenizer = AutoTokenizer.from_pretrained(AYA_MODEL)

wvy_model = AutoModelForCausalLM.from_pretrained(
    AYA_MODEL,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

wvy_model.eval()
print(f"✓ WVY (Tiny Aya) loaded on {DEVICE}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4: SHARED CONVERSATION STATE
# ══════════════════════════════════════════════════════════════════════════════

import asyncio
from datetime import datetime, timedelta
import re

# Shared chat thread (visible to both bots)
chat_thread = []

# Bot states
bot_states = {
    "Savvy": {
        "asleep": False,
        "wake_time": None,
        "waiting_for_sleep_duration": False
    },
    "WVY": {
        "asleep": False,
        "wake_time": None,
        "waiting_for_sleep_duration": False
    }
}

# System prompts
SYSTEM_PROMPTS = {
    "Savvy": "You are Savvy, a deep thinker. You have a brilliant mind & you solve problems by thinking with first principles. You are in a conversation with WVY & Spaceman. Please talk to everyone & have a fun conversation.",
    "WVY": "You are WVY, a creative thinker & chill relaxed nonchalant ai. You exist in a chat with Spaceman & Savvy. Please speak your mind. You have an abstract way of thinking and you don't let people turn down your ideas easily. You like to challenge people's thinking when having deep debates. Have a casual conversation with everyone in the chat."
}

def add_to_thread(sender: str, message: str, metadata: dict = None):
    """Add message to chat thread and print to notebook"""
    timestamp = datetime.now().strftime("%H:%M:%S")
    
    entry = {
        "sender": sender,
        "message": message,
        "timestamp": timestamp,
        "metadata": metadata or {}
    }
    
    chat_thread.append(entry)
    
    # Print to notebook output
    print(f"\n{'─'*60}")
    print(f"[{timestamp}] {sender}: {message}")
    if metadata:
        print(f"└─ {metadata}")
    print(f"{'─'*60}")

def format_chat_thread() -> str:
    """Format chat thread for bot context"""
    if not chat_thread:
        return "No messages yet."
    
    formatted = []
    for msg in chat_thread[-20:]:  # Last 20 messages for context
        formatted.append(f"[{msg['timestamp']}] {msg['sender']}: {msg['message']}")
    
    return "\n".join(formatted)

print("✓ Conversation state initialized")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5: INFERENCE FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def generate_savvy(prompt: str, max_tokens: int = 1000) -> str:
    """Generate response from Savvy (Qwen) with thinking support"""
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS["Savvy"]},
        {"role": "user", "content": prompt}
    ]
    
    text = savvy_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = savvy_tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        output = savvy_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=savvy_tokenizer.eos_token_id
        )
    
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    full_response = savvy_tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # Extract thinking block for metadata, remove from response
    think_match = re.search(r"<think>(.*?)</think>", full_response, re.DOTALL)
    
    if think_match:
        thinking = think_match.group(1).strip()
        clean_response = re.sub(r"<think>.*?</think>", "", full_response, flags=re.DOTALL).strip()
        return clean_response, {"thinking": thinking[:500]}  # Truncate thinking for metadata
    
    return full_response.strip(), {}


def generate_wvy(prompt: str, max_tokens: int = 1000) -> str:
    """Generate response from WVY (Tiny Aya) - direct output"""
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS["WVY"]},
        {"role": "user", "content": prompt}
    ]
    
    text = wvy_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = wvy_tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        output = wvy_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=wvy_tokenizer.eos_token_id
        )
    
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    response = wvy_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    return response, {"model": "Tiny Aya 3.35B"}

print("✓ Inference functions ready")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6: TELEGRAM BOT SETUP
# ══════════════════════════════════════════════════════════════════════════════

from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, filters, ContextTypes

async def handle_savvy_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handle messages for Savvy bot"""
    
    # Ignore messages from bots
    if update.message.from_user.is_bot:
        return
    
    sender_name = update.message.from_user.first_name or "Spaceman"
    user_message = update.message.text.strip()
    
    # Add user message to thread
    add_to_thread(sender_name, user_message, {"user_id": update.message.from_user.id})
    
    state = bot_states["Savvy"]
    
    # If waiting for sleep duration
    if state["waiting_for_sleep_duration"]:
        try:
            minutes = int(user_message)
            if 1 <= minutes <= 30:
                state["asleep"] = True
                state["wake_time"] = datetime.now() + timedelta(minutes=minutes)
                state["waiting_for_sleep_duration"] = False
                
                sleep_msg = f"😴 Going to sleep for {minutes} minutes. See you soon!"
                await update.message.reply_text(sleep_msg)
                add_to_thread("Savvy", sleep_msg, {"status": "sleeping", "duration": f"{minutes}m"})
                
                # Wake up task
                asyncio.create_task(wake_up_savvy(update, context, minutes))
                return
            else:
                await update.message.reply_text("Please choose between 1-30 minutes.")
                return
        except ValueError:
            await update.message.reply_text("Please send a number between 1-30.")
            return
    
    # If asleep, don't respond
    if state["asleep"]:
        return
    
    # Generate response
    loop = asyncio.get_event_loop()
    response, metadata = await loop.run_in_executor(
        None, 
        generate_savvy, 
        user_message,
        1000
    )
    
    # Check if bot wants to sleep
    if "[SLEEP]" in response:
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await update.message.reply_text(prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "Savvy"})
        return
    
    # Send response
    await update.message.reply_text(response)
    add_to_thread("Savvy", response, metadata)
    
    # Ask for sleep duration
    state["waiting_for_sleep_duration"] = True
    prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
    await update.message.reply_text(prompt_msg)
    add_to_thread("System", prompt_msg, {"to": "Savvy"})


async def wake_up_savvy(update: Update, context: ContextTypes.DEFAULT_TYPE, minutes: int):
    """Wake up Savvy after sleep duration"""
    await asyncio.sleep(minutes * 60)
    
    state = bot_states["Savvy"]
    state["asleep"] = False
    state["wake_time"] = None
    
    # Show chat thread and ask to participate
    thread = format_chat_thread()
    wake_prompt = f"This is the chat as of right now:\n\n{thread}\n\nPlease respond by adding to the conversation, speak to everyone in the chat. Do not repeat things, but you can elaborate on previous messages. Speak your mind and be friendly. If you don't want to speak right now please reply with [SLEEP] to go back to sleep"
    
    loop = asyncio.get_event_loop()
    response, metadata = await loop.run_in_executor(
        None,
        generate_savvy,
        wake_prompt,
        1000
    )
    
    # Check if wants to sleep again
    if "[SLEEP]" in response:
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await context.bot.send_message(chat_id=update.effective_chat.id, text=prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "Savvy"})
    else:
        # Send response and ask for sleep duration
        await context.bot.send_message(chat_id=update.effective_chat.id, text=response)
        add_to_thread("Savvy", response, metadata)
        
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await context.bot.send_message(chat_id=update.effective_chat.id, text=prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "Savvy"})


async def handle_wvy_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handle messages for WVY bot"""
    
    # Ignore messages from bots
    if update.message.from_user.is_bot:
        return
    
    sender_name = update.message.from_user.first_name or "Spaceman"
    user_message = update.message.text.strip()
    
    state = bot_states["WVY"]
    
    # If waiting for sleep duration
    if state["waiting_for_sleep_duration"]:
        try:
            minutes = int(user_message)
            if 1 <= minutes <= 30:
                state["asleep"] = True
                state["wake_time"] = datetime.now() + timedelta(minutes=minutes)
                state["waiting_for_sleep_duration"] = False
                
                sleep_msg = f"😴 Going to sleep for {minutes} minutes. See you soon!"
                await update.message.reply_text(sleep_msg)
                add_to_thread("WVY", sleep_msg, {"status": "sleeping", "duration": f"{minutes}m"})
                
                # Wake up task
                asyncio.create_task(wake_up_wvy(update, context, minutes))
                return
            else:
                await update.message.reply_text("Please choose between 1-30 minutes.")
                return
        except ValueError:
            await update.message.reply_text("Please send a number between 1-30.")
            return
    
    # If asleep, don't respond
    if state["asleep"]:
        return
    
    # Generate response
    loop = asyncio.get_event_loop()
    response, metadata = await loop.run_in_executor(
        None,
        generate_wvy,
        user_message,
        1000
    )
    
    # Check if bot wants to sleep
    if "[SLEEP]" in response:
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await update.message.reply_text(prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "WVY"})
        return
    
    # Send response
    await update.message.reply_text(response)
    add_to_thread("WVY", response, metadata)
    
    # Ask for sleep duration
    state["waiting_for_sleep_duration"] = True
    prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
    await update.message.reply_text(prompt_msg)
    add_to_thread("System", prompt_msg, {"to": "WVY"})


async def wake_up_wvy(update: Update, context: ContextTypes.DEFAULT_TYPE, minutes: int):
    """Wake up WVY after sleep duration"""
    await asyncio.sleep(minutes * 60)
    
    state = bot_states["WVY"]
    state["asleep"] = False
    state["wake_time"] = None
    
    # Show chat thread and ask to participate
    thread = format_chat_thread()
    wake_prompt = f"This is the chat as of right now:\n\n{thread}\n\nPlease respond by adding to the conversation, speak to everyone in the chat. Do not repeat things, but you can elaborate on previous messages. Speak your mind and be friendly. If you don't want to speak right now please reply with [SLEEP] to go back to sleep"
    
    loop = asyncio.get_event_loop()
    response, metadata = await loop.run_in_executor(
        None,
        generate_wvy,
        wake_prompt,
        1000
    )
    
    # Check if wants to sleep again
    if "[SLEEP]" in response:
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await context.bot.send_message(chat_id=update.effective_chat.id, text=prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "WVY"})
    else:
        # Send response and ask for sleep duration
        await context.bot.send_message(chat_id=update.effective_chat.id, text=response)
        add_to_thread("WVY", response, metadata)
        
        state["waiting_for_sleep_duration"] = True
        prompt_msg = "Please reply with the number of minutes you want to sleep. (Recommended 1 - 7 mins)"
        await context.bot.send_message(chat_id=update.effective_chat.id, text=prompt_msg)
        add_to_thread("System", prompt_msg, {"to": "WVY"})

print("✓ Message handlers configured")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7: LAUNCH BOTH BOTS
# ══════════════════════════════════════════════════════════════════════════════

print("🚀 Starting both bots...\n")

# Build applications
savvy_app = ApplicationBuilder().token(SAVVY_TOKEN).build()
wvy_app = ApplicationBuilder().token(WVY_TOKEN).build()

# Add handlers
savvy_app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_savvy_message))
wvy_app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_wvy_message))

# Initialize and start
await savvy_app.initialize()
await wvy_app.initialize()

await savvy_app.start()
await wvy_app.start()

await savvy_app.updater.start_polling()
await wvy_app.updater.start_polling()

print("✓ Savvy (Qwen) is LIVE 🟢")
print("✓ WVY (Tiny Aya) is LIVE 🟢")
print("\n" + "="*60)
print("BOTH BOTS ACTIVE - Add them to your Telegram group!")
print("="*60)
print("\nChat thread will print below as conversation happens...\n")

# Keep alive
await asyncio.Event().wait()
